# Motivating example

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/finite_element_method/motivating_example.ipynb)

Consider an ODE $y^\prime = -y$, $y(0) = 1$, which has an analytical answer $y(x) = e^{-x}$.

We wish to approximate this function numerically between $[0, 1]$ with a line defined by the boundary points, $f(0) = a = 1$ and $f(1) = b$:

$$f(x) = [1 - x] + x b$$

How do we find $b$?

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Option 1: Direct substitution

Following the Finite Difference method, we can substitute $f(1)$ into the ODE and solve for $b$:

$$\begin{aligned}
f'(1) &= -f(1) \\
b - 1 &= -b \\
b &= \frac{1}{2}
\end{aligned}$$

and the approximate solution is $f(x) = 1 - \frac{x}{2}$.

In [ ]:
f = lambda x: 1 - x / 2
y = lambda x: np.exp(-x)

x = np.linspace(0, 1, 100)
f_values = f(x)
y_values = y(x)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=f_values, name="f(x) = 1 - x/2"))
fig.add_trace(go.Scatter(x=x, y=y_values, name="y(x) = e^(-x)"))
fig.update_layout(
    title="Plot of f(x) and y(x) from 0 to 1",
    xaxis_title="x",
    yaxis_title="Value",
    template="plotly_white",
)
fig.show()

## Option 2: Least square minimization of the residual

Let's integrate the squared residual of $f(x)$ plugged into the ODE, $y^\prime + y = 0$:

$$\begin{aligned}
R &= f^\prime(x) + f(x) \\
&= [b - 1] + 1 - x + x b \\
&= b + x[b - 1] \\
R^2 &= b^2 + 2 b x[b - 1] + [x[b - 1]]^2 \\
F &= \int_0^1 R^2 \, \mathrm{d}x = \frac{1}{3} [7 b^2 - 5 b + 1]
\end{aligned}$$

Note that in the end the integral of the residual, $F$, is a parabolic function of the parameter $b$. The minimum is found at $\frac{\partial F}{\partial b}(b) = 0$, for $b = \frac{5}{14}$. The approximate solution is $f(x) = 1 - \frac{9}{14}x$.

In [ ]:
f_new = lambda x: 1 - (9 / 14) * x
res_opt1 = lambda x: -0.5 + f(x)
res_opt2 = lambda x: -9 / 14 + f_new(x)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Solutions on [0, 1]", "Residual R(x) = f'(x) + f(x)"),
)
fig.add_trace(go.Scatter(x=x, y=f_values, name="f(x) = 1 - x/2"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=y_values, name="y(x) = e^(-x)"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=f_new(x), name="f(x) = 1 - 9/14 x", line=dict(dash="dash")), row=1, col=1)

fig.add_trace(go.Scatter(x=x, y=res_opt1(x), name="Residual (1 - x/2)"), row=1, col=2)
fig.add_trace(go.Scatter(x=x, y=res_opt2(x), name="Residual (1 - 9/14 x)", line=dict(dash="dash")), row=1, col=2)
fig.update_xaxes(title_text="x")
fig.update_yaxes(title_text="Value", row=1, col=1)
fig.update_yaxes(title_text="Residual", row=1, col=2)
fig.update_layout(template="plotly_white", height=420)
fig.show()

## Method of Weighted Residuals (MWR)

We see that Option 2 does a better job of approximating the function over the full domain since it focuses on the integral of the residual.

You may have noticed that we did something funny here: we integrated $R^2$ and then took its derivative with respect to $b$ in order to find the minimum of the integral! I.e.:

$$\begin{aligned}
\frac{\partial F}{\partial b} = 0 &= \frac{\partial}{\partial b} \int_0^1 R^2 \, \mathrm{d}x \\
&= \int_0^1 2R \frac{\partial R}{\partial b} \, \mathrm{d}x \\
&= \int_0^1 R v \, \mathrm{d}x
\end{aligned}$$

where the residual has now been weighted by the function $v$. This is called the Method of Weighted Residuals (MWR). Let's try a different function and see what happens.

## Option 3: Integrate the MWR to find the parameters

Integrate the weighted residual function directly with a convenient choice of $v$.

Let's take $v = x$ (to be motivated later) and integrate:

$$\begin{aligned}
0 &= \int_0^1 R x \, \mathrm{d}x \\
&= \int_0^1 \big[b + [b - 1]x\big] x \, \mathrm{d}x \\
&= \int_0^1 \big[b x + [b - 1]x^2\big] \, \mathrm{d}x \\
&= \bigg[\frac{1}{6} x^2 \big[3b + 2[b - 1]x\big]\bigg]_0^1 \\
&= \frac{5}{6}b - \frac{1}{3} \implies b = \frac{2}{5}
\end{aligned}$$

which is very close to our previous solution $b = \frac{5}{14}$, but with a simpler integration. BUT, we are still left with analytically integrating! If only there were a way to express integrals as the sum of the integrand evaluated at certain points...

## Option 4: Express the MWR integral as the sum of the integrand evaluated at certain points

Recall Gaussian Quadrature allows us to evaluate an integral by summing the integrand at the Gauss Points. The Gauss Points for the domain $[-1, 1]$ are $\pm\frac{1}{\sqrt{3}}$, which scaled to this integral domain $[0, 1]$ become (with Jacobian factor $\frac{1}{2}$):

$$\begin{aligned}
x_{[0,1]} &= 0.5 (x_{[-1,1]} + 1) \\
&= \frac{1}{2} \bigg[1 \pm \frac{1}{\sqrt{3}} \bigg] \\
&\approx [0.21132487, \, 0.78867513]
\end{aligned}$$

and

$$\begin{aligned}
0 &= \int_0^1 R x \, \mathrm{d}x \\
&= \frac{1}{2} \big[(R x)\big|_{x = 0.21132487} + (R x)\big|_{x = 0.78867513}\big] \\
&= \frac{1}{2} (-0.0446582 + 0.255983 b - 0.622008 + 1.41068 b) \\
b &= \frac{2}{5}
\end{aligned}$$

## Recap of the approach

Let's recap what we've done:

1. Parameterized a function as a weighted sum of simpler functions (a linear basis)
2. Found a (simple) integral expression that minimizes the error in the approximation (Method of Weighted Residuals)
3. Performed the integration *exactly* using only function evaluations into a linear system (Gaussian quadrature)
4. Solved the linear system (I **told** you everything boiled down to linear systems!)

Practical solution of linear systems requires sparsity! Let's formalize our procedure and see how we can ensure sparsity.